In [4]:
import pandas as pd
import numpy as np
from lightgbm import LGBMClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import classification_report, roc_auc_score
import pickle, shap, warnings
warnings.filterwarnings('ignore')

# ── 1. LOAD ───────────────────────────────────────────────────────────────────
df   = pd.read_csv('../data/daily_fitbit_sema_df_unprocessed.csv')
stai = pd.read_csv('../data/stai.csv', index_col=0)
panas = pd.read_csv('../data/panas.csv', index_col=0)

# ── 2. CLEAN & PARSE ──────────────────────────────────────────────────────────
df['date'] = pd.to_datetime(df['date'])
stai['submitdate'] = pd.to_datetime(stai['submitdate'])
panas['submitdate'] = pd.to_datetime(panas['submitdate'])

# Sleep duration from ms to hours
df['sleep_duration_hrs'] = df['sleep_duration'] / 3_600_000

# Drop rows with no mood label at all
mood_cols = ['TENSE/ANXIOUS','HAPPY','ALERT','NEUTRAL','RESTED/RELAXED','SAD','TIRED']
df = df.dropna(subset=['TENSE/ANXIOUS'])
df['TENSE/ANXIOUS'] = df['TENSE/ANXIOUS'].astype(int)

# ── 3. MERGE STAI + PANAS (weekly surveys → forward fill to daily) ─────────────
# For each user, forward-fill STAI score across days until next submission
def merge_weekly_survey(df, survey, value_col, new_col):
    survey = survey.rename(columns={'submitdate': 'date'})
    survey = survey[['id', 'date', value_col]].copy()
    
    merged = pd.merge_asof(
        df.sort_values('date'),
        survey[['id', 'date', value_col]].sort_values('date'),
        on='date',
        by='id',
        direction='backward',   # use most recent survey before this date
        tolerance=pd.Timedelta('14d')  # only use if within 2 weeks
    )
    merged = merged.rename(columns={value_col: new_col})
    return merged

df = merge_weekly_survey(df, stai.rename(columns={'user_id':'id'}), 
                         'stai_stress', 'stai_score')
df = merge_weekly_survey(df, panas.rename(columns={'user_id':'id'}), 
                         'negative_affect_score', 'negative_affect')

# ── 4. PER-USER ROLLING BASELINES ─────────────────────────────────────────────
df = df.sort_values(['id', 'date'])

def rolling_baseline(df, col, window=7):
    return df.groupby('id')[col].transform(
        lambda x: x.shift(1).rolling(window, min_periods=3).mean()
    )

def rolling_slope(df, col, window=3):
    def slope(x):
        if len(x) < 2: return 0
        return np.polyfit(range(len(x)), x, 1)[0]
    return df.groupby('id')[col].transform(
        lambda x: x.shift(1).rolling(window, min_periods=2).apply(slope)
    )

df['rmssd_7d']         = rolling_baseline(df, 'rmssd')
df['hr_7d']            = rolling_baseline(df, 'resting_hr')
df['sleep_7d']         = rolling_baseline(df, 'sleep_duration_hrs')

df['rmssd_deviation']  = df['rmssd'] - df['rmssd_7d']
df['hr_deviation']     = df['resting_hr'] - df['hr_7d']
df['sleep_deviation']  = df['sleep_duration_hrs'] - df['sleep_7d']
df['sleep_trend_3d']   = rolling_slope(df, 'sleep_duration_hrs')
df['rmssd_trend_3d']   = rolling_slope(df, 'rmssd')

# ── 5. FEATURE LIST ───────────────────────────────────────────────────────────
FEATURES = [
    'rmssd', 'rmssd_deviation', 'rmssd_trend_3d', 'rmssd_7d',
    'nremhr', 'resting_hr', 'hr_deviation',
    'nightly_temperature', 'daily_temperature_variation',
    'full_sleep_breathing_rate',
    'sleep_duration_hrs', 'sleep_deviation', 'sleep_trend_3d',
    'sleep_efficiency', 'sleep_deep_ratio', 'sleep_rem_ratio',
    'sleep_wake_ratio', 'minutesToFallAsleep', 'minutesAwake',
    'steps', 'sedentary_minutes', 'very_active_minutes',
    'lightly_active_minutes', 'moderately_active_minutes',
    'calories', 'spo2',
    'stai_score', 'negative_affect',   # weekly surveys forward-filled
]

# Only keep rows where we have enough features
df_model = df.dropna(subset=['rmssd', 'resting_hr', 'sleep_duration_hrs', 'TENSE/ANXIOUS'])
df_model = df_model.copy()

X = df_model[FEATURES].values
y = df_model['TENSE/ANXIOUS'].values
groups = df_model['id'].values

print(f"Dataset: {len(df_model)} rows, {y.mean():.1%} positive (tense/anxious)")

# ── 6. TRAIN / VAL SPLIT BY USER ──────────────────────────────────────────────
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, val_idx = next(gss.split(X, y, groups))

X_train, X_val = X[train_idx], X[val_idx]
y_train, y_val = y[train_idx], y[val_idx]

# ── 7. TRAIN ──────────────────────────────────────────────────────────────────
model = LGBMClassifier(
    n_estimators=500,
    learning_rate=0.03,
    max_depth=5,
    num_leaves=24,
    min_child_samples=15,
    subsample=0.8,
    colsample_bytree=0.8,
    class_weight='balanced',    # handles imbalance — more ok days than bad
    random_state=42,
    verbose=-1
)

model.fit(
    X_train, y_train,
    feature_name=FEATURES,
    eval_set=[(X_val, y_val)],
)

# ── 8. EVALUATE ───────────────────────────────────────────────────────────────
y_pred = model.predict(X_val)
y_prob = model.predict_proba(X_val)[:, 1]

print(classification_report(y_val, y_pred, target_names=['OK', 'Tense/Anxious']))
print(f"ROC-AUC: {roc_auc_score(y_val, y_prob):.3f}")

# ── 9. SHAP ───────────────────────────────────────────────────────────────────
explainer = shap.TreeExplainer(model)

# ── 10. SAVE ──────────────────────────────────────────────────────────────────
bundle = {
    'model': model,
    'features': FEATURES,
    'explainer': explainer,
}
with open('model.pkl', 'wb') as f:
    pickle.dump(bundle, f)

print("Saved model.pkl")

Dataset: 1170 rows, 22.8% positive (tense/anxious)
               precision    recall  f1-score   support

           OK       0.88      0.83      0.85       173
Tense/Anxious       0.36      0.44      0.40        36

     accuracy                           0.77       209
    macro avg       0.62      0.64      0.62       209
 weighted avg       0.79      0.77      0.78       209

ROC-AUC: 0.667
Saved model.pkl
